# Lab 03 · Baseline Extraction (Notebook Walkthrough)

This notebook runs the **same pipeline the web UI runs**, in-process, so you can watch ingestion happen, inspect the chunks it produces, and query the index — all with captured outputs.

**Concept.** The `baseline_extract` profile uses only Azure AI Search's built-in `DocumentExtractionSkill`. No semantic chunking tuning, no generative enrichment, no visual/NLP analysis yet. This is the starting retrieval quality every later lab improves on.

Source document for the whole run-through: **Deep Excavation Design and Construction.pdf**.

## Step 1 — Bootstrap the backend

`bootstrap()` loads the gitignored `.env`, puts the repo on `sys.path`, and confirms the live Azure configuration.

In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR if (NB_DIR / 'lab_runtime.py').exists() else NB_DIR / 'notebooks'))
import lab_runtime as lab

info = lab.bootstrap()
info

{'repo_root': '<home>\\rag-on-azure-lab',
 'env_values_loaded': 89,
 'search_endpoint': 'https://your-search-service.search.windows.net',
 'search_configured': True,
 'embedding_deployment': 'text-embedding-3-large-vector',
 'chat_deployment': 'gpt-5-4-mini-chat',
 'agentic_planning_model': 'gpt-5-4-mini-search',
 'canonical_index': 'ai-search-lab-index'}

## Step 2 — Ingest with the baseline profile

Full pipeline: **parse → normalize → chunk → Azure AI Search blob + skillset enrichment → publish**. The first run drives the real Azure indexer (a few minutes); re-runs reuse the cached result.

In [2]:
job = lab.ingest(skill_profile='baseline_extract', reuse=True)
lab.chunk_overview(job)

Reusing existing 'baseline_extract' ingestion (doc_id=eb402ae0, 412 chunks). Pass reuse=False to force a fresh run.


{'doc_id': 'eb402ae0536243e0a7539ef590fb41e6',
 'skill_profile': 'baseline_extract',
 'chunk_count': 412,
 'avg_tokens': 200.4,
 'max_tokens': 777,
 'chunks_with_summary': 0,
 'chunks_with_keyword_hints': 0,
 'chunks_with_image_description': 0}

`chunks_with_summary`, `chunks_with_keyword_hints`, and `chunks_with_image_description` are all **0** — baseline extraction produces plain text chunks with no enrichment.

## Step 3 — Inspect the raw chunks

In [3]:
import pandas as pd

chunks = lab.load_chunks(job)
pd.DataFrame([
    {
        'section': ' > '.join(c.get('section_path') or []) or '(root)',
        'pages': c.get('page_numbers'),
        'tokens': c.get('token_estimate'),
        'text_preview': (c.get('clean_text') or '')[:120].replace(chr(10), ' '),
    }
    for c in chunks[:8]
])

,section,pages,tokens,text_preview
0,Deep Excavation Design and Construction,[1],34,Geotechnical Engineering Office Civil Engineer...
1,Deep Excavation Design and Construction,"[2, 3]",107,Geotechnical Engineering Office Civil Engineer...
2,Top Left,[3],21,Excavation using an Interlocking Pipe Pile Wal...
3,Top Right,[3],21,Excavation using a Tied-back Wall for a Reside...
4,Bottom Left,[3],16,Excavation using a Diaphragm Wall for a Kai Ta...
5,Bottom Right,[3],32,Excavation using a Multi-cell Circular Shaft f...
6,Foreword,[4],582,The GCO Publication No. 1/90 - Review of Desig...
7,Foreword,[4],469,This publication was prepared under the overal...


## Step 4 — Query with Full-Text retrieval

Full-text (BM25) shines when the answer is **explicitly stated** and keyword-heavy.

In [4]:
Q1 = 'What are the objectives of site investigation for ELS works?'

resp = lab.ask(Q1, job=job, retrieval_mode='full_text', record_as='lab03_baseline_fulltext')
lab.show_answer(resp)

[retrieval_mode=full_text | scoring_profile=default | citations=8]

The main objectives of site investigation are to acquire knowledge of site characteristics that affect such works and plan for their safe execution, with due consideration given to the nearby buildings/structures/services. When planning ELS works, site investigation should normally include a detailed desk study, site reconnaissance, topographic survey, groun [1]

Supporting evidence:
- The various stages of site investigation, design and construction of ELS works require coordinated input from experienced designers and contractors. Continuous involvement of the designer of the ELS works is essential for verifying both the validity of the assumed ground conditions and the expected performance of the ELS works. [5]
- Good practice for site investigation and selection of geotechnical parameters that are crucial for the design of ELS works and associated key considerations are presented in Chapter 2. A review of common typ

### Inspect the raw scored hits

The actual Azure AI Search `@search.score` ordering that produced the answer:

In [5]:
hits = lab.retrieve(Q1, job=job, retrieval_mode='full_text', top=5)
lab.hits_table(hits)

,score,reranker,section,pages,snippet
0,31.9041,None,2.1 Site Investigation,[17],The main objectives of site investigation are ...
1,12.3033,None,1.2 Overview,"[15, 16]",Good practice for site investigation and selec...
2,11.9138,None,9.2.2.3 Empirical Limits for Ground Settlement,[114],Action Level 3 is intended to flag up the situ...
3,11.8185,None,2.1.1 Ground Investigation,[17],GI should be properly planned in order to deve...
4,11.7842,None,1.3 General Guidance,[16],"The various stages of site investigation, desi..."


## Takeaways

- Baseline extraction gives **searchable text and structure** but no enrichment fields.
- Full-text retrieval already answers precise, terminology-heavy questions well because question wording matches document wording.
- Later labs ask the **same** question against richer indexes so you can see what enrichment adds.

Next: **Lab 04** adds chunking + vectorization, unlocking `vector` and `hybrid` retrieval and scoring profiles.